# Hyperparameter Search — Five Regression Models

This notebook provides a unified hyperparameter-selection workflow for Random Forest (RF), K-Nearest Neighbors (KNN), Decision Tree (DT), Multilayer Perceptron (MLP), and Support Vector Regression (SVR).

**Methodological scope:** The SVR grid and 3-fold cross-validation below are supported by preserved experiment configuration. The RF/KNN/DT/MLP grids below are **reconstructed search spaces that include the reported final settings**, not verified copies of their original optimization searches. Running them can produce different winning parameters. The training/evaluation notebook uses the documented selected configurations and does not depend on these searches being rerun.

All searches use only the 12 development airfoil geometries; the four unseen geometries are excluded.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR

DATA_CANDIDATES = [Path('data/PALMO_NACA_4Series_Complete_Dataset.xlsx'), Path('../data/PALMO_NACA_4Series_Complete_Dataset.xlsx')]
DATA_FILE = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_FILE is None:
    raise FileNotFoundError('Run this notebook from the repository root or notebooks/ folder.')
df = pd.read_excel(DATA_FILE, sheet_name='PALMO_4Series_Data')
features = ['%Camber', '%Thickness', 'Reynolds#', 'Mach#', 'Alpha']
targets = ['cl', 'cd', 'cm']
test_mask = (((df['%Camber']==3) & (df['%Thickness'].isin([15,18]))) | ((df['%Camber']==4) & (df['%Thickness'].isin([15,21]))))
X_dev = df.loc[~test_mask, features]
y_dev = df.loc[~test_mask, targets]
print('Development rows:', len(X_dev), 'Held-out rows:', int(test_mask.sum()))

In [ ]:
# The four non-SVR search spaces are reproducible reconstructions, not recovered original grids.
# Each includes the configuration recorded in the final engineering report.
searches = {
    'RF': {
        'estimator': RandomForestRegressor(random_state=42, n_jobs=-1),
        'grid': {'n_estimators':[150,300,450], 'max_depth':[10,20,None], 'min_samples_leaf':[1,2], 'min_samples_split':[2,4], 'max_features':[1.0, 'sqrt']},
        'n_iter':16, 'folds':10,
    },
    'KNN': {
        'estimator': Pipeline([('scaler',StandardScaler()), ('model', KNeighborsRegressor(n_jobs=-1))]),
        'grid': {'model__n_neighbors':[3,5,7], 'model__weights':['distance','uniform'], 'model__p':[1,2], 'model__leaf_size':[20,30,40]},
        'n_iter':12, 'folds':10,
    },
    'DT': {
        'estimator': DecisionTreeRegressor(random_state=42),
        'grid': {'max_depth':[10,20,None], 'min_samples_leaf':[1,2,4], 'min_samples_split':[2,4], 'splitter':['best','random'], 'ccp_alpha':[0.0,0.001]},
        'n_iter':16, 'folds':10,
    },
    'MLP': {
        'estimator': Pipeline([('scaler',StandardScaler()), ('model',MLPRegressor(random_state=42, max_iter=500))]),
        'grid': {'model__hidden_layer_sizes':[(128,64),(256,128)], 'model__alpha':[0.0001,0.001], 'model__learning_rate_init':[0.003,0.001], 'model__activation':['relu'], 'model__solver':['adam']},
        'n_iter':8, 'folds':10,
    },
    'SVR': {
        'estimator': Pipeline([('scale',StandardScaler()), ('svr',TransformedTargetRegressor(regressor=MultiOutputRegressor(SVR(kernel='rbf',cache_size=256,tol=0.001,max_iter=-1), n_jobs=1),transformer=StandardScaler()))]),
        'grid': {'svr__regressor__estimator__C':[1.0,10.0], 'svr__regressor__estimator__epsilon':[0.01,0.05], 'svr__regressor__estimator__gamma':[0.1,0.2]},
        'n_iter':None, 'folds':3,
    }
}
print('Configured tuning candidates:', ', '.join(searches))

In [ ]:
# Run one model at a time; tuning can be computationally expensive.
MODEL_TO_TUNE = 'RF'  # RF, KNN, DT, MLP, or SVR
RUN_TUNING = False  # Set True to perform the search

if RUN_TUNING:
    key = MODEL_TO_TUNE.upper()
    config = searches[key]
    cv = KFold(n_splits=config['folds'], shuffle=True, random_state=42)
    common = dict(estimator=config['estimator'], scoring='r2', cv=cv, n_jobs=1, refit=True, verbose=1)
    if key == 'SVR':
        search = GridSearchCV(param_grid=config['grid'], **common)
    else:
        search = RandomizedSearchCV(param_distributions=config['grid'], n_iter=config['n_iter'], random_state=42, **common)
    search.fit(X_dev, y_dev)
    output_dir=Path('results') if Path('data').exists() else Path('../results')
    output_dir.mkdir(exist_ok=True)
    pd.DataFrame(search.cv_results_).to_csv(output_dir/f'{key}_hyperparameter_search.csv',index=False)
    (output_dir/f'{key}_selected_parameters.json').write_text(json.dumps({'model':key,'best_params':search.best_params_,'cv_r2':float(search.best_score_)},indent=2))
    print(key, search.best_params_, search.best_score_)
else:
    print('Search not executed. Set RUN_TUNING = True when you want to tune a selected model.')